# Airbnb Listings Data Cleaning

This notebook prepares the Seattle Airbnb listings data for the pricing analysis. The workflow focuses on creating a cleaner and more comparable sample of short-term Seattle listings and exporting a cleaned file for downstream analysis.

## Key cleaning decisions

- Filter to listings with host location in Seattle
- Drop columns that are mostly administrative, text heavy, duplicate, or not needed for the analysis
- Convert boolean, percent, price, and date fields to usable data types
- Remove rows with missing `price`
- Restrict to more comparable short-term listings with `minimum_nights <= 7` and `accommodates >= 2`
- Create `normalized_price` as price per guest


## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

RAW_PATH = "../data/listings.csv"
CLEAN_PATH = "../data/listings_cleaned.csv"

listings = pd.read_csv(RAW_PATH)
print(f"Raw shape: {listings.shape}")
listings.head()

## 2. Initial inspection

Start with a quick check of the raw data structure before filtering and cleaning.


In [ ]:
listings.info()

## 3. Filter to Seattle hosts

The broader file may include listings outside the intended market. Restrict the sample to listings where `host_location` contains Seattle.


In [ ]:
listings = listings[
    (listings["host_location"].notna()) &
    (listings["host_location"].str.contains("seattle", case=False))
].copy()

print(f"Shape after Seattle filter: {listings.shape}")
listings.head()

## 4. Drop columns not needed for analysis

These fields are mostly URLs, text descriptions, duplicate geographic fields, or low value administrative columns that are unlikely to be useful for the pricing analysis.


In [ ]:
cols_to_drop = [
    "listing_url",
    "scrape_id",
    "last_scraped",
    "source",
    "name",
    "description",
    "neighborhood_overview",
    "picture_url",
    "host_url",
    "host_name",
    "host_about",
    "host_thumbnail_url",
    "host_picture_url",
    "host_verifications",
    "host_response_time",
    "host_neighbourhood",
    "host_listings_count",
    "latitude",
    "longitude",
    "neighbourhood",
    "bathrooms_text",
    "calendar_updated",
    "calendar_last_scraped",
    "number_of_reviews_l30d",
    "estimated_occupancy_l365d",
    "estimated_revenue_l365d",
    "first_review",
    "last_review",
    "license",
]

listings_cleaned = listings.drop(columns=cols_to_drop).copy()
print(f"Shape after dropping columns: {listings_cleaned.shape}")
listings_cleaned.head()

## 5. Clean data types

Convert indicator columns to boolean values, parse percentage fields as numeric values, convert `price` to numeric, and cast `host_since` as a datetime field.


In [ ]:
bool_cols = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "instant_bookable",
]

bool_map = {"t": True, "f": False}
for col in bool_cols:
    listings_cleaned[col] = listings_cleaned[col].map(bool_map)

pct_cols = ["host_response_rate", "host_acceptance_rate"]
for col in pct_cols:
    listings_cleaned[col] = pd.to_numeric(
        listings_cleaned[col].str.replace("%", "", regex=False),
        errors="coerce",
    )

listings_cleaned["price"] = pd.to_numeric(
    listings_cleaned["price"].str.replace(r"[$,]", "", regex=True),
    errors="coerce",
)

listings_cleaned["host_since"] = pd.to_datetime(
    listings_cleaned["host_since"],
    errors="coerce",
)

listings_cleaned.dtypes

## 6. Review missingness

Check which fields still contain missing values after the basic cleaning steps.


In [ ]:
missing_summary = (
    listings_cleaned.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

missing_summary[missing_summary["missing_count"] > 0]

`price` is central to the analysis, so listings missing `price` should be removed. Other fields with missing values can be handled later depending on the specific analysis.


In [ ]:
listings_cleaned = listings_cleaned.dropna(subset=["price"]).copy()
print(f"Shape after dropping missing price: {listings_cleaned.shape}")

## 7. Review extreme prices

Inspect the highest priced listings before deciding how to make the sample more comparable.


In [ ]:
listings_cleaned[["price", "accommodates", "minimum_nights"]].sort_values(
    "price", ascending=False
).head(10)

In [ ]:
listings_cleaned.boxplot(
    column="price",
    by="neighbourhood_group_cleansed",
    rot=45,
)

plt.suptitle("")
plt.title("Price Distribution by Neighborhood Group")
plt.show()

## 8. Restrict to comparable short-term listings

Very high prices are often driven by listings with long minimum stay requirements or by listings meant for a single guest, which are less comparable to typical short-term nightly rentals.

To focus the sample, keep only:
- listings with `minimum_nights <= 7`
- listings with `accommodates >= 2`


In [ ]:
listings_cleaned = listings_cleaned[
    (listings_cleaned["minimum_nights"] <= 7)
    & (listings_cleaned["accommodates"] >= 2)
].copy()

print(f"Shape after comparability filters: {listings_cleaned.shape}")

## 9. Feature engineering

Create `normalized_price` as price per guest so listings of different sizes are more comparable.


In [ ]:
listings_cleaned["normalized_price"] = (
    listings_cleaned["price"] / listings_cleaned["accommodates"]
)

listings_cleaned[["price", "accommodates", "normalized_price"]].sort_values(
    "normalized_price", ascending=False
).head(10)

## 10. Check cleaned distributions

In [ ]:
listings_cleaned.boxplot(
    column="price",
    by="neighbourhood_group_cleansed",
    rot=45,
)

plt.suptitle("")
plt.title("Price Distribution by Neighborhood Group")
plt.show()

In [ ]:
listings_cleaned.boxplot(
    column="normalized_price",
    by="neighbourhood_group_cleansed",
    rot=45,
)

plt.suptitle("")
plt.title("Normalized Price Distribution by Neighborhood Group")
plt.show()

## 11. Export cleaned data

In [ ]:
listings_cleaned.to_csv(CLEAN_PATH, index=False)
print(f"Cleaned file exported to {CLEAN_PATH}")